Step 2 – Upload the datasets

In [1]:
from google.colab import files

uploaded = files.upload()

Saving portuguese_test_processed.csv to portuguese_test_processed.csv
Saving portuguese_train_processed.csv to portuguese_train_processed.csv


Step 3 – Import libraries

In [2]:
import numpy as np
import pandas as pd

from pathlib import Path
from IPython.display import display

from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor

from sklearn.model_selection import (
    KFold,
    cross_validate,
    cross_val_predict
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

print("All libraries imported successfully!")

All libraries imported successfully!


Step 4 – Set up project folders

In [3]:
project_dir = Path("/content/Portuguese_ML_Project")

models_dir = project_dir / "models"

results_dir = (
    project_dir / "output" /
    "random_forest" / "comparison"
)

models_dir.mkdir(parents=True, exist_ok=True)
results_dir.mkdir(parents=True, exist_ok=True)

print("Project folders created!")
print("Results location:", results_dir)

Project folders created!
Results location: /content/Portuguese_ML_Project/output/random_forest/comparison


Step 5 – Load processed training and testing data

In [4]:
train_df = pd.read_csv(
    "/content/portuguese_train_processed.csv"
)

test_df = pd.read_csv(
    "/content/portuguese_test_processed.csv"
)

X_train = train_df.drop(columns=["G3"])
y_train = train_df["G3"]

X_test = test_df.drop(columns=["G3"])
y_test = test_df["G3"]

print("Training Shape:", X_train.shape)
print("Testing Shape:", X_test.shape)

print("Training Target:", y_train.shape)
print("Testing Target:", y_test.shape)

assert X_train.columns.equals(X_test.columns)
assert X_train.notna().all().all()
assert y_train.notna().all()
assert X_test.notna().all().all()
assert y_test.notna().all()

display(train_df.head())

Training Shape: (519, 58)
Testing Shape: (130, 58)
Training Target: (519,)
Testing Target: (130,)


,num__age,num__Medu,num__Fedu,num__traveltime,num__studytime,num__failures,num__famrel,num__freetime,num__goout,num__Dalc,...,cat__activities_yes,cat__nursery_no,cat__nursery_yes,cat__higher_no,cat__higher_yes,cat__internet_no,cat__internet_yes,cat__romantic_no,cat__romantic_yes,G3
0,0.987932,-0.437418,-0.256337,-0.754310,1.352962,-0.382133,0.068136,-0.186680,-0.196270,-0.539674,...,0.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,18
1,-0.629534,1.350140,1.591423,-0.754310,0.121054,-0.382133,0.068136,0.753970,1.501468,3.579309,...,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,12
2,0.987932,0.456361,-0.256337,-0.754310,1.352962,-0.382133,1.108214,-0.186680,-1.045139,-0.539674,...,1.0,1.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,12
3,0.179199,-0.437418,-1.180216,-0.754310,-1.110853,-0.382133,0.068136,0.753970,-1.045139,0.490072,...,0.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,13
4,0.179199,-1.331196,-1.180216,0.555011,-1.110853,-0.382133,-0.971942,1.694619,1.501468,0.490072,...,1.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,10


Step 6 – Define Random Forest experiments

In [5]:
rf_experiments = {
    "RF-1": {
        "n_estimators": 100,
        "max_depth": None,
        "min_samples_leaf": 1
    },

    "RF-2": {
        "n_estimators": 200,
        "max_depth": 5,
        "min_samples_leaf": 1
    },

    "RF-3": {
        "n_estimators": 200,
        "max_depth": 10,
        "min_samples_leaf": 2
    },

    "RF-4": {
        "n_estimators": 300,
        "max_depth": 15,
        "min_samples_leaf": 2
    },

    "RF-5": {
        "n_estimators": 300,
        "max_depth": None,
        "min_samples_leaf": 4
    }
}

experiment_df = pd.DataFrame.from_dict(
    rf_experiments,
    orient="index"
)

experiment_df.index.name = "Experiment"

display(experiment_df.reset_index())

,Experiment,n_estimators,max_depth,min_samples_leaf
0,RF-1,100,NaN,1
1,RF-2,200,5.0,1
2,RF-3,200,10.0,2
3,RF-4,300,15.0,2
4,RF-5,300,NaN,4


Step 7 – Configure five-fold cross-validation

In [6]:
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "MAE": "neg_mean_absolute_error",
    "MSE": "neg_mean_squared_error",
    "R2": "r2"
}

print("5-Fold Cross-Validation configured!")

5-Fold Cross-Validation configured!


Step 8 – Train and evaluate all Random Forest configurations

In [7]:
rf_results = []
rf_fold_results = []
rf_predictions = {}

actual = y_train.to_numpy()
actual_binary = (actual >= 10).astype(int)

for name, params in rf_experiments.items():

    print(f"Training {name}...")

    model = RandomForestRegressor(
        **params,
        random_state=42,
        n_jobs=-1
    )

    # 5-fold cross-validation
    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        return_train_score=True,
        n_jobs=1,
        error_score="raise"
    )

    # Out-of-fold predictions
    predictions = cross_val_predict(
        model,
        X_train,
        y_train,
        cv=cv,
        n_jobs=1
    )

    rf_predictions[name] = predictions

    # Rounded grades, limited to 0–20
    rounded_predictions = np.clip(
        np.rint(predictions), 0, 20
    ).astype(int)

    # Pass/fail classification
    predicted_binary = (
        predictions >= 10
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        actual_binary,
        predicted_binary,
        labels=[0, 1]
    ).ravel()

    # Mean fold RMSE
    train_rmse = np.sqrt(
        -scores["train_MSE"]
    ).mean()

    val_rmse = np.sqrt(
        -scores["test_MSE"]
    ).mean()

    rf_results.append({
        "Experiment": name,

        "Trees": params["n_estimators"],
        "Max Depth": str(params["max_depth"]),
        "Min Leaf": params["min_samples_leaf"],

        # Regression metrics
        "Train MAE": -scores["train_MAE"].mean(),
        "Validation MAE": -scores["test_MAE"].mean(),

        "Train MSE": -scores["train_MSE"].mean(),
        "Validation MSE": -scores["test_MSE"].mean(),

        "Train RMSE": train_rmse,
        "Validation RMSE": val_rmse,

        "Train R2": scores["train_R2"].mean(),
        "Validation R2": scores["test_R2"].mean(),

        "RMSE Gap": val_rmse - train_rmse,

        # Grade prediction accuracy
        "Exact Grade Accuracy (%)": (
            np.mean(
                rounded_predictions == actual
            ) * 100
        ),

        "Within 1 Grade (%)": (
            np.mean(
                np.abs(actual - predictions) <= 1
            ) * 100
        ),

        "Within 2 Grades (%)": (
            np.mean(
                np.abs(actual - predictions) <= 2
            ) * 100
        ),

        # Classification metrics
        "Pass/Fail Accuracy (%)": (
            accuracy_score(
                actual_binary,
                predicted_binary
            ) * 100
        ),

        "Precision": precision_score(
            actual_binary,
            predicted_binary,
            zero_division=0
        ),

        "Recall": recall_score(
            actual_binary,
            predicted_binary,
            zero_division=0
        ),

        "Specificity": (
            tn / (tn + fp)
            if (tn + fp) > 0 else 0
        ),

        "F1 Score": f1_score(
            actual_binary,
            predicted_binary,
            zero_division=0
        ),

        # Confusion matrix
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp)
    })

    # Store fold-level metrics
    for fold in range(5):

        rf_fold_results.append({
            "Experiment": name,
            "Fold": fold + 1,

            "Train MAE": -scores["train_MAE"][fold],
            "Validation MAE": -scores["test_MAE"][fold],

            "Train MSE": -scores["train_MSE"][fold],
            "Validation MSE": -scores["test_MSE"][fold],

            "Train RMSE": np.sqrt(
                -scores["train_MSE"][fold]
            ),

            "Validation RMSE": np.sqrt(
                -scores["test_MSE"][fold]
            ),

            "Train R2": scores["train_R2"][fold],
            "Validation R2": scores["test_R2"][fold]
        })

rf_results_df = pd.DataFrame(rf_results)

rf_results_df = rf_results_df.sort_values(
    by=[
        "Validation RMSE",
        "Validation MAE",
        "RMSE Gap"
    ],
    ascending=[True, True, True]
).reset_index(drop=True)

rf_fold_df = pd.DataFrame(rf_fold_results)

print("\nAll Random Forest experiments completed!")

display(rf_results_df.round(4))

Training RF-1...
Training RF-2...
Training RF-3...
Training RF-4...
Training RF-5...

All Random Forest experiments completed!


,Experiment,Trees,Max Depth,Min Leaf,Train MAE,Validation MAE,Train MSE,Validation MSE,Train RMSE,Validation RMSE,...,Within 2 Grades (%),Pass/Fail Accuracy (%),Precision,Recall,Specificity,F1 Score,TN,FP,FN,TP
0,RF-5,300,None,4,0.5303,0.8518,0.7575,1.7397,0.8683,1.2993,...,93.6416,89.7881,0.9751,0.9009,0.8824,0.9365,75,10,43,391
1,RF-4,300,15,2,0.3919,0.8437,0.4360,1.7514,0.6580,1.3041,...,94.0270,90.1734,0.9799,0.9009,0.9059,0.9388,77,8,43,391
2,RF-3,200,10,2,0.4038,0.8474,0.4431,1.7682,0.6635,1.3126,...,93.8343,89.5954,0.9750,0.8986,0.8824,0.9353,75,10,44,390
3,RF-2,200,5,1,0.5921,0.8509,0.6984,1.8283,0.8339,1.3348,...,93.2563,89.7881,0.9751,0.9009,0.8824,0.9365,75,10,43,391
4,RF-1,100,None,1,0.3185,0.8745,0.2497,1.8971,0.4985,1.3587,...,92.2929,88.8247,0.9796,0.8848,0.9059,0.9298,77,8,50,384


Step 9 – Regression error comparison

In [8]:
regression_columns = [
    "Experiment",
    "Trees",
    "Max Depth",
    "Min Leaf",
    "Train MAE",
    "Validation MAE",
    "Train MSE",
    "Validation MSE",
    "Train RMSE",
    "Validation RMSE",
    "Train R2",
    "Validation R2",
    "RMSE Gap"
]

print("=" * 65)
print("RANDOM FOREST – REGRESSION ERROR COMPARISON")
print("=" * 65)

display(
    rf_results_df[regression_columns].round(4)
)

RANDOM FOREST – REGRESSION ERROR COMPARISON


,Experiment,Trees,Max Depth,Min Leaf,Train MAE,Validation MAE,Train MSE,Validation MSE,Train RMSE,Validation RMSE,Train R2,Validation R2,RMSE Gap
0,RF-5,300,None,4,0.5303,0.8518,0.7575,1.7397,0.8683,1.2993,0.9281,0.8323,0.4310
1,RF-4,300,15,2,0.3919,0.8437,0.4360,1.7514,0.6580,1.3041,0.9587,0.8301,0.6461
2,RF-3,200,10,2,0.4038,0.8474,0.4431,1.7682,0.6635,1.3126,0.9580,0.8278,0.6492
3,RF-2,200,5,1,0.5921,0.8509,0.6984,1.8283,0.8339,1.3348,0.9337,0.8220,0.5009
4,RF-1,100,None,1,0.3185,0.8745,0.2497,1.8971,0.4985,1.3587,0.9763,0.8142,0.8602


Step 10 – Grade prediction accuracy comparison

In [9]:
grade_columns = [
    "Experiment",
    "Exact Grade Accuracy (%)",
    "Within 1 Grade (%)",
    "Within 2 Grades (%)"
]

print("=" * 65)
print("RANDOM FOREST – GRADE ACCURACY COMPARISON")
print("=" * 65)

display(
    rf_results_df[grade_columns].round(2)
)

RANDOM FOREST – GRADE ACCURACY COMPARISON


,Experiment,Exact Grade Accuracy (%),Within 1 Grade (%),Within 2 Grades (%)
0,RF-5,42.20,73.03,93.64
1,RF-4,43.74,73.41,94.03
2,RF-3,44.12,72.45,93.83
3,RF-2,43.55,73.41,93.26
4,RF-1,42.58,70.91,92.29


Step 11 – Pass/fail classification comparison

In [10]:
classification_columns = [
    "Experiment",
    "Pass/Fail Accuracy (%)",
    "Precision",
    "Recall",
    "Specificity",
    "F1 Score"
]

print("=" * 65)
print("RANDOM FOREST – CLASSIFICATION COMPARISON")
print("=" * 65)

display(
    rf_results_df[classification_columns].round(4)
)

RANDOM FOREST – CLASSIFICATION COMPARISON


,Experiment,Pass/Fail Accuracy (%),Precision,Recall,Specificity,F1 Score
0,RF-5,89.7881,0.9751,0.9009,0.8824,0.9365
1,RF-4,90.1734,0.9799,0.9009,0.9059,0.9388
2,RF-3,89.5954,0.9750,0.8986,0.8824,0.9353
3,RF-2,89.7881,0.9751,0.9009,0.8824,0.9365
4,RF-1,88.8247,0.9796,0.8848,0.9059,0.9298


Step 12 – Confusion matrix comparison

In [11]:
confusion_columns = [
    "Experiment",
    "TN",
    "FP",
    "FN",
    "TP"
]

print("=" * 65)
print("RANDOM FOREST – CONFUSION MATRIX COMPARISON")
print("=" * 65)

display(
    rf_results_df[confusion_columns]
)

RANDOM FOREST – CONFUSION MATRIX COMPARISON


,Experiment,TN,FP,FN,TP
0,RF-5,75,10,43,391
1,RF-4,77,8,43,391
2,RF-3,75,10,44,390
3,RF-2,75,10,43,391
4,RF-1,77,8,50,384


Step 13 – Check overfitting and underfitting

In [12]:
baseline_model = DummyRegressor(
    strategy="mean"
)

baseline_scores = cross_validate(
    baseline_model,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring
)

baseline_rmse = np.sqrt(
    -baseline_scores["test_MSE"]
).mean()

def assess_fitting(row):

    gap_ratio = (
        row["RMSE Gap"] / row["Train RMSE"]
        if row["Train RMSE"] > 0
        else np.inf
    )

    if row["Validation RMSE"] >= baseline_rmse:
        return "Weak performance / possible underfitting"

    elif gap_ratio > 0.30:
        return "Possible overfitting"

    else:
        return "Reasonable generalization"

rf_results_df["Fitting Assessment"] = (
    rf_results_df.apply(
        assess_fitting,
        axis=1
    )
)

print("=" * 65)
print("RANDOM FOREST – FITTING ASSESSMENT")
print("=" * 65)

print(f"Baseline Validation RMSE: {baseline_rmse:.4f}")

display(
    rf_results_df[
        [
            "Experiment",
            "Train RMSE",
            "Validation RMSE",
            "RMSE Gap",
            "Fitting Assessment"
        ]
    ].round(4)
)

RANDOM FOREST – FITTING ASSESSMENT
Baseline Validation RMSE: 3.2326


,Experiment,Train RMSE,Validation RMSE,RMSE Gap,Fitting Assessment
0,RF-5,0.8683,1.2993,0.4310,Possible overfitting
1,RF-4,0.6580,1.3041,0.6461,Possible overfitting
2,RF-3,0.6635,1.3126,0.6492,Possible overfitting
3,RF-2,0.8339,1.3348,0.5009,Possible overfitting
4,RF-1,0.4985,1.3587,0.8602,Possible overfitting


Step 14 – Display five-fold performance

In [13]:
print("=" * 65)
print("RANDOM FOREST – FIVE-FOLD PERFORMANCE")
print("=" * 65)

display(
    rf_fold_df.sort_values(
        ["Experiment", "Fold"]
    ).round(4)
)

RANDOM FOREST – FIVE-FOLD PERFORMANCE


,Experiment,Fold,Train MAE,Validation MAE,Train MSE,Validation MSE,Train RMSE,Validation RMSE,Train R2,Validation R2
0,RF-1,1,0.3187,0.8325,0.2609,1.6503,0.5108,1.2847,0.9764,0.8025
1,RF-1,2,0.3341,0.8532,0.2918,1.9836,0.5402,1.4084,0.9739,0.7394
2,RF-1,3,0.3163,0.8704,0.2657,1.7588,0.5155,1.3262,0.9733,0.8620
3,RF-1,4,0.2961,1.0654,0.1933,3.0164,0.4397,1.7368,0.9798,0.7821
4,RF-1,5,0.3273,0.7513,0.2366,1.0762,0.4865,1.0374,0.9781,0.8851
5,RF-2,1,0.5939,0.8094,0.7540,1.5788,0.8683,1.2565,0.9318,0.8110
6,RF-2,2,0.6046,0.8109,0.7285,1.7828,0.8535,1.3352,0.9349,0.7657
7,RF-2,3,0.5930,0.8313,0.7529,1.7226,0.8677,1.3125,0.9244,0.8649
8,RF-2,4,0.5632,1.0408,0.5263,2.9496,0.7254,1.7174,0.9449,0.7869
9,RF-2,5,0.6057,0.7619,0.7302,1.1076,0.8545,1.0524,0.9325,0.8817


Step 15 – Final complete Random Forest comparison

In [14]:
final_comparison = rf_results_df.copy()

final_comparison.insert(
    0,
    "Rank",
    np.arange(1, len(final_comparison) + 1)
)

print("=" * 70)
print("FINAL PORTUGUESE RANDOM FOREST COMPARISON")
print("=" * 70)

with pd.option_context(
    "display.max_columns", None,
    "display.width", 250
):
    display(final_comparison.round(4))

FINAL PORTUGUESE RANDOM FOREST COMPARISON


,Rank,Experiment,Trees,Max Depth,Min Leaf,Train MAE,Validation MAE,Train MSE,Validation MSE,Train RMSE,Validation RMSE,Train R2,Validation R2,RMSE Gap,Exact Grade Accuracy (%),Within 1 Grade (%),Within 2 Grades (%),Pass/Fail Accuracy (%),Precision,Recall,Specificity,F1 Score,TN,FP,FN,TP,Fitting Assessment
0,1,RF-5,300,None,4,0.5303,0.8518,0.7575,1.7397,0.8683,1.2993,0.9281,0.8323,0.4310,42.1965,73.0250,93.6416,89.7881,0.9751,0.9009,0.8824,0.9365,75,10,43,391,Possible overfitting
1,2,RF-4,300,15,2,0.3919,0.8437,0.4360,1.7514,0.6580,1.3041,0.9587,0.8301,0.6461,43.7380,73.4104,94.0270,90.1734,0.9799,0.9009,0.9059,0.9388,77,8,43,391,Possible overfitting
2,3,RF-3,200,10,2,0.4038,0.8474,0.4431,1.7682,0.6635,1.3126,0.9580,0.8278,0.6492,44.1233,72.4470,93.8343,89.5954,0.9750,0.8986,0.8824,0.9353,75,10,44,390,Possible overfitting
3,4,RF-2,200,5,1,0.5921,0.8509,0.6984,1.8283,0.8339,1.3348,0.9337,0.8220,0.5009,43.5453,73.4104,93.2563,89.7881,0.9751,0.9009,0.8824,0.9365,75,10,43,391,Possible overfitting
4,5,RF-1,100,None,1,0.3185,0.8745,0.2497,1.8971,0.4985,1.3587,0.9763,0.8142,0.8602,42.5819,70.9056,92.2929,88.8247,0.9796,0.8848,0.9059,0.9298,77,8,50,384,Possible overfitting


Step 16 – Select the best Random Forest configuration

In [15]:
best_result = final_comparison.iloc[0]

best_name = best_result["Experiment"]
best_params = rf_experiments[best_name]

print("=" * 60)
print("BEST PORTUGUESE RANDOM FOREST CONFIGURATION")
print("=" * 60)

print("Experiment:", best_name)
print("Parameters:", best_params)
print("Number of Features:", X_train.shape[1])

print("\nREGRESSION METRICS")
print(f"Validation RMSE : {best_result['Validation RMSE']:.4f}")
print(f"Validation MAE  : {best_result['Validation MAE']:.4f}")
print(f"Validation R²   : {best_result['Validation R2']:.4f}")
print(f"RMSE Gap        : {best_result['RMSE Gap']:.4f}")

print("\nGRADE ACCURACY")
print(
    f"Exact Accuracy  : "
    f"{best_result['Exact Grade Accuracy (%)']:.2f}%"
)
print(
    f"Within ±1 Grade : "
    f"{best_result['Within 1 Grade (%)']:.2f}%"
)
print(
    f"Within ±2 Grades: "
    f"{best_result['Within 2 Grades (%)']:.2f}%"
)

print("\nPASS/FAIL CLASSIFICATION")
print(
    f"Accuracy   : "
    f"{best_result['Pass/Fail Accuracy (%)']:.2f}%"
)
print(f"Precision  : {best_result['Precision']:.4f}")
print(f"Recall     : {best_result['Recall']:.4f}")
print(f"Specificity: {best_result['Specificity']:.4f}")
print(f"F1 Score   : {best_result['F1 Score']:.4f}")

print("\nCONFUSION MATRIX")
print("TN:", int(best_result["TN"]))
print("FP:", int(best_result["FP"]))
print("FN:", int(best_result["FN"]))
print("TP:", int(best_result["TP"]))

print("\nFITTING ASSESSMENT")
print(best_result["Fitting Assessment"])

BEST PORTUGUESE RANDOM FOREST CONFIGURATION
Experiment: RF-5
Parameters: {'n_estimators': 300, 'max_depth': None, 'min_samples_leaf': 4}
Number of Features: 58

REGRESSION METRICS
Validation RMSE : 1.2993
Validation MAE  : 0.8518
Validation R²   : 0.8323
RMSE Gap        : 0.4310

GRADE ACCURACY
Exact Accuracy  : 42.20%
Within ±1 Grade : 73.03%
Within ±2 Grades: 93.64%

PASS/FAIL CLASSIFICATION
Accuracy   : 89.79%
Precision  : 0.9751
Recall     : 0.9009
Specificity: 0.8824
F1 Score   : 0.9365

CONFUSION MATRIX
TN: 75
FP: 10
FN: 43
TP: 391

FITTING ASSESSMENT
Possible overfitting
